# Fabric Audit Data Collection

This notebook collects audit data from Microsoft Fabric for governance purposes.

## What this notebook does:
- Authenticates to Fabric API
- Retrieves activity logs for a specified time period
- Exports audit data to Excel/CSV for analysis

In [ ]:
# Import required modules
import sys
sys.path.append('..')

from modules.fabric_auth import FabricAuth, load_credentials
from modules.fabric_client import FabricClient
from modules.utils import generate_audit_report, export_to_excel, export_to_csv
from datetime import datetime, timedelta
import pandas as pd

In [ ]:
# Load credentials and authenticate
credentials = load_credentials('../config/credentials.json')
auth = FabricAuth(
    tenant_id=credentials['tenant_id'],
    client_id=credentials['client_id'],
    client_secret=credentials['client_secret']
)

token = auth.get_access_token()
if token:
    print("✓ Authentication successful")
    client = FabricClient(token)
else:
    print("✗ Authentication failed")

In [ ]:
# Define the time period for audit data collection
end_date = datetime.now()
start_date = end_date - timedelta(days=7)  # Last 7 days

start_datetime = start_date.strftime('%Y-%m-%dT%H:%M:%S')
end_datetime = end_date.strftime('%Y-%m-%dT%H:%M:%S')

print(f"Collecting audit data from {start_datetime} to {end_datetime}")

In [ ]:
# Retrieve activity events
print("Fetching activity events...")
events = client.get_activity_events(start_datetime, end_datetime)

if events:
    print(f"✓ Retrieved {len(events)} activity events")
else:
    print("✗ No events retrieved or error occurred")

In [ ]:
# Generate audit report
if events:
    audit_df = generate_audit_report(events)
    print(f"\nAudit Report Summary:")
    print(f"Total events: {len(audit_df)}")
    print(f"\nFirst few records:")
    display(audit_df.head(10))

In [ ]:
# Analyze audit data
if events and not audit_df.empty:
    print("\n=== Audit Data Analysis ===")
    
    # Count by operation type
    if 'Operation' in audit_df.columns:
        print("\nTop Operations:")
        print(audit_df['Operation'].value_counts().head(10))
    
    # Count by user
    if 'UserId' in audit_df.columns:
        print("\nMost Active Users:")
        print(audit_df['UserId'].value_counts().head(10))

In [ ]:
# Export audit data
if events and not audit_df.empty:
    # Export to Excel
    output_filename = f"fabric_audit_{start_date.strftime('%Y%m%d')}_{end_date.strftime('%Y%m%d')}.xlsx"
    export_to_excel(audit_df, output_filename, sheet_name='Audit Data')
    
    # Optionally export to CSV
    # csv_filename = f"fabric_audit_{start_date.strftime('%Y%m%d')}_{end_date.strftime('%Y%m%d')}.csv"
    # export_to_csv(audit_df, csv_filename)
    
    print(f"\n✓ Audit data exported successfully")